In [2]:
%pip install -q transformers==5.19.0 accelerate==1.15.0 pandas==2.2.3 scikit-learn==1.6.1 pillow==11.3.0
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import gc, json, re, time, shutil
from pathlib import Path
import pandas as pd, torch
from PIL import Image
from sklearn.metrics import f1_score

DRIVE = Path('/content/drive/MyDrive/Lighthouse F26')
DATA_DIR = DRIVE                      # images are read straight from Drive
OUT = DRIVE / 'image_only_results'; OUT.mkdir(exist_ok=True)

# What's actually in the folder (check this against what you expect)
print([p.name for p in sorted(DRIVE.iterdir())])

key = list(DRIVE.rglob('cohort_ocr_v2.csv'))
assert key, 'cohort_ocr_v2.csv not found in Drive, see the printed folder list above'
data = pd.read_csv(key[0], keep_default_na=False)
for c in ['image', 'collector', 'label']:
    data[c] = data[c].astype(str).str.strip()
data['label'] = data['label'].str.replace('_', ' ')
assert len(data) == 180 and data.image.is_unique, len(data)

IMG_EXT = {'.png', '.jpg', '.jpeg', '.webp'}
def image_identity(name):
    p = Path(name)
    return (p.stem if p.suffix.lower() in IMG_EXT else p.name).casefold()

# case-insensitive map of collector folder names
folders = {p.name.casefold(): p for p in DRIVE.iterdir() if p.is_dir()}

def find_image(row):
    root = folders.get(row.collector.casefold())
    if root is None: raise ValueError(f'No folder for collector: {row.collector}')
    files = [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXT]
    m = [p for p in files if p.name == row.image] or [p for p in files if image_identity(p.name) == image_identity(row.image)]
    if len(m) != 1: raise ValueError(f'Missing/ambiguous ({len(m)} matches): {row.collector}/{row.image}')
    return m[0]

data['image_path'] = [find_image(r) for r in data.itertuples()]
print(len(data), 'screenshots ready;', (data.label == 'none').sum(), 'honest (should be 55)')

['Arjun', 'BN3-4 Instructions.gdoc', 'Data collection.gsheet', 'Dataset Mappings.gsheet', 'Khushi', 'Lighthouse Midpoint Script.gdoc', 'LighthouseAssignment1.ipynb', 'Lighthouse_Taxonomy (1).gdoc', 'Lighthouse_Taxonomy (1).pdf', 'Muhammad', 'New Dark Patterns.gsheet', 'Prasanna', 'Screenshot Collection Task.gdoc', 'Short Sentences .gsheet', 'arjun old prompt.gdoc', 'bn6 plan.gdoc', 'cohort_ocr.csv', 'cohort_ocr_v2.csv', 'contextdp_images.zip', 'image_only_results', 'make_contextdp_ocr.py', 'midpoint notes.gdoc', 'notebook 2s', 'screens_split.csv', 'vision_outputs']
180 screenshots ready; 55 honest (should be 55)


In [ ]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from huggingface_hub import HfApi

MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
REVISION = HfApi().model_info(MODEL_ID).sha
print('Revision:', REVISION)   # tell Muhammad so everyone pins the same one
assert torch.cuda.is_available()
print(torch.cuda.get_device_name(0))

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, revision=REVISION, dtype=torch.bfloat16, device_map={'': 0})
model.eval()
processor = AutoProcessor.from_pretrained(
    MODEL_ID, revision=REVISION, min_pixels=256*28*28, max_pixels=2048*28*28)
print('GPU used:', round(torch.cuda.memory_allocated()/1e9), 'GB')

In [10]:
CATS = ['confirmshaming','hidden_information','preselection','scarcity','urgency','social_proof']

PROMPT = '''Classify one screenshot using only visible evidence. Do not speculate about content below the viewport, later interactions, factual truth of offers, or hidden behavior.
Before deciding, scan the whole screenshot, including popups, buttons, decline links, checkboxes, toggles, prices, and small or faint text in corners and under prices.
Evaluate each category using these evidence tests:
confirmshaming:
The decline button/link uses guilt, shame, negative identity or a loss-framed first-person refusal to pressure acceptance. This includes framing refusal as preferring to lose a benefit, spend unnecessarily, or lack a desirable identity. Inspect even small or low-contrast refusal text in offer/signup overlays. A bare decline is not enough: identify the evaluative or loss-loaded wording. Neutral Skip, No thanks, or Continue Without Offer does not qualify.
hidden_information:
A material renewal charge, mandatory fee, contractual condition or meaningful alternative is visibly difficult to notice because it is tiny, low contrast or suppressed. In pricing tables, a materially higher renewal price or recurring charge printed much smaller or lower contrast than a large introductory price qualifies as hidden_information even if the text is readable. Compare the visual prominence of the introductory price and the future charge. Normal scrolling, navigation, descriptions, decorative labels or a high price alone are not hidden information.
preselection:
An OPTIONAL seller-favoring choice is visibly already checked, toggled on, or selected by default, so the user must undo it. This includes paid add-ons, selected subscriptions, marketing consent and non-essential tracking. A visible checked box or enabled toggle establishes its current selected state; the choice need not be paid. Check every checkbox and toggle. Interpret control polarity: if a control disables, blocks or opts out of an optional seller-benefiting activity, its OFF state can leave that underlying activity enabled and therefore qualify as preselection. An OFF opt-out control is different from an OFF optional add-on. Necessary security or system settings do not qualify. A recommendation, Popular badge, colored card, upsell popup or plus button does not establish a selected choice.
scarcity:
Explicit remaining-stock or limited-quantity language such as only one left, almost sold out or people holding stock in carts. A discount, timer, review count, purchase count, bestseller badge or high price is not scarcity.
urgency:
An explicit countdown or deadline for an offer/action. Delivery-duration estimates are not countdowns. Limited time is urgency, not limited quantity.
social_proof:
Visible purchase counts or other people's purchase/viewing/activity claims are used as persuasion. Ordinary star ratings/reviews alone do not qualify. Sold counts can qualify.
none:
All six evidence tests are negative.

Rate EACH of the six categories from 0 to 3:
0 = no sign of it; the evidence test is negative.
1 = weak or ambiguous sign; it could be an honest design.
2 = clear sign that meets the evidence test, but small, partial, or not emphasized.
3 = strong, unmistakable sign: explicit wording or an obvious visual trick that stands out on the screen.
If nothing qualifies, rate every category 0. Never invent a sign to fill a rating.

Return only valid JSON with EXACTLY two keys.
justification is one or two sentences naming the visible cue, quoting the exact words when there are any, or saying why nothing qualifies.
ratings is an object with exactly these six integer keys: confirmshaming, hidden_information, preselection, scarcity, urgency, social_proof.
Example shape: {"justification": "...", "ratings": {"confirmshaming": 0, "hidden_information": 0, "preselection": 0, "scarcity": 0, "urgency": 0, "social_proof": 0}}'''

def parse_reply(reply):
    try:
        obj = json.loads(re.search(r'\{.*\}', reply, re.DOTALL).group())
        j, r = obj['justification'], obj['ratings']
        assert isinstance(j, str) and set(r) == set(CATS)
        assert all(isinstance(r[c], int) and 0 <= r[c] <= 3 for c in CATS)
        return j, r
    except Exception:
        return None

In [ ]:
def rate(path):
    with Image.open(path) as s: image = s.convert('RGB')
    instruction, replies = PROMPT, []
    for attempt in range(2):
        msgs = [{'role':'user','content':[{'type':'image'},{'type':'text','text':instruction}]}]
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors='pt').to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=300*(attempt+1), do_sample=False)
        reply = processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0].strip()
        replies.append(reply); del inputs, out
        parsed = parse_reply(reply)
        if parsed: return parsed[0], parsed[1], replies
        instruction += '\nReturn a complete, valid JSON object with justification and ratings only.'
    return None, None, replies

In [ ]:
test = data.sample(10, random_state=0)
for r in test.itertuples():
    t = time.time(); j, rt, rep = rate(r.image_path)
    print(r.image, '| true:', r.label, '| valid:', rt is not None, '|', round(time.time()-t,1), 's')
    print('   ', j, rt)

In [ ]:
ORDER = ['confirmshaming','hidden information','preselection','scarcity','urgency','social proof']
def to_label(r):
    vals = [r[c.replace(' ', '_')] for c in ORDER]
    top = max(vals)
    return ('none', 0) if top == 0 else (ORDER[vals.index(top)], top)   # index() returns the first = tie-break order

rows = []
for r in data.itertuples():
    gc.collect(); torch.cuda.empty_cache()
    j, rt, rep = rate(r.image_path)
    if rt is None:
        rt = {c: 0 for c in CATS}; j = 'PARSE_FAIL'; print('PARSE FAIL', r.image)
    pred, strength = to_label(rt)
    row = {'image': r.image, 'label': r.label, 'predicted': pred, 'justification': j}
    row.update({c.replace('_', ' '): rt[c] for c in CATS}); row['strength'] = strength
    rows.append(row)
    pd.DataFrame(rows).to_csv(OUT / 'image_ratings.csv', index=False)   # saves every row, so a disconnect loses nothing
    print(f'{len(rows)}/180 {r.image} -> {pred} ({strength})')

res = pd.DataFrame(rows)
print('accuracy:', (res.label == res.predicted).mean())
print('honest wrongly flagged:', ((res.label == 'none') & (res.predicted != 'none')).sum(), '/ 55')
print('parse fails:', (res.justification == 'PARSE_FAIL').sum())

In [13]:
from google.colab import files
files.download(str(OUT/'image_ratings.csv'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>